# **01 Data Loading**

In [7]:
import pandas as pd

# load each dataset and print the data range it covers
files = {
    "cases": "datasets/cases_malaysia.csv",
    "tests": "datasets/tests_malaysia.csv",
    "vax": "datasets/vax_malaysia.csv",
    "hospital": "datasets/hospital.csv",
    "icu": "datasets/icu.csv",
    "population": "datasets/population.csv",
}

dfs = {}
for name, path in files.items():
    df = pd.read_csv(path)
    dfs[name] = df
    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        print(f"{name:12s} rows={len(df):6d} date range={df['date'].min().date()} to {df['date'].max().date()}")
    else:
        print(f"{name:12s} rows={len(df):6d} no date column")

print()

for name, df in dfs.items():
    if "date" in df.columns:
        dupes = df["date"].duplicated().sum()
        print(f"{name:12s} {dupes} duplicate dates")

cases        rows=  1954 date range=2020-01-25 to 2025-05-31
tests        rows=  1955 date range=2020-01-24 to 2025-05-31
vax          rows=  1460 date range=2021-02-24 to 2025-02-22
hospital     rows= 29765 date range=2020-03-24 to 2025-05-31
icu          rows= 29605 date range=2020-03-24 to 2025-05-31
population   rows=    18 no date column

cases        0 duplicate dates
tests        0 duplicate dates
vax          0 duplicate dates
hospital     27874 duplicate dates
icu          27714 duplicate dates


Most files cover about 2020 to 2025 daily. The vax file starts later, on 2021-02-24, since that is when Malaysia's vaccination program began and population has no date column since it is a static per-state table used later for normalizing. Hospital and icu have far more rows than the rest as they include one row per state per day, so they will need to be filtered down to just the "Malaysia" row before merging later on.

In [8]:
# list the unique state values in hospital, icu and population
hospital = dfs["hospital"]
icu = dfs["icu"]
population = dfs["population"]

print("hospital states:", sorted(hospital["state"].unique()), "\n")
print("icu states:", sorted(icu["state"].unique()), "\n")
print("population states:", sorted(population["state"].unique()), "\n")

# confirm a "Malaysia" row exists for every date in hospital and icu
n_dates = hospital["date"].nunique()
malaysia_rows = (hospital["state"] == "Malaysia").sum()

print(f"hospital: {n_dates} unique dates, {malaysia_rows} Malaysia rows -> {'match' if n_dates == malaysia_rows else 'MISMATCH'}")

n_dates = icu["date"].nunique()
malaysia_rows = (icu["state"] == "Malaysia").sum()
print(f"icu: {n_dates} unique dates, {malaysia_rows} Malaysia rows -> {'match' if n_dates == malaysia_rows else 'MISMATCH'}")

hospital states: ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya'] 

icu states: ['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya'] 

population states: ['Johor', 'Kedah', 'Kelantan', 'Klang Valley', 'Malaysia', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya'] 

hospital: 1891 unique dates, 0 Malaysia rows -> MISMATCH
icu: 1891 unique dates, 0 Malaysia rows -> MISMATCH


hospital.csv and icu.csv have no national "Malaysia" row, only the 16 states/territories, across all 1891 dates. Therefore, national totals must be computed by summing all 16 state rows per date in step 4. population.csv does have a "Malaysia" row, but also a "Klang Valley" row, which is a multi-state region, not a real state, so it should be excluded from any state-level summation.

In [9]:
# aggregate hospital and icu to national totals by summing all states per date
hospital_national = hospital.groupby("date", as_index=False).sum(numeric_only=True)
icu_national = icu.groupby("date", as_index=False).sum(numeric_only=True)

# merge all national-level daily tables together on date
merged = dfs["cases"].merge(dfs["tests"], on="date", how="outer")
merged = merged.merge(dfs["vax"], on="date", how="outer")
merged = merged.merge(hospital_national, on="date", how="outer")
merged = merged.merge(icu_national, on="date", how="outer")
merged = merged.sort_values("date").reset_index(drop=True)


print(f"merged shape: {merged.shape}")
print(f"date range: {merged['date'].min().date()} -> {merged['date'].max().date()}")
merged.head()

merged shape: (1955, 108)
date range: 2020-01-24 -> 2025-05-31


,date,cases_new,cases_import,cases_recovered,cases_active,cases_cluster,cases_unvax,cases_pvax,cases_fvax,cases_boost,...,vent,vent_port,icu_covid,icu_pui,icu_noncovid,vent_covid,vent_pui,vent_noncovid,vent_used,vent_port_used
0,2020-01-24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-25,4.0,4.0,0.0,4.0,0.0,4.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-26,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-27,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-28,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


The merge produced 1955 rows spanning 2020-01-24 to 2025-05-31, one row per date, with 108 columns from all five files combined. NaNs appear early on because some files start later than others, for example icu and hospital data only begins 2020-03-24 and vax only begins 2021-02-24. These gaps will be handled in Phase 2 (cleaning), not filled in now.